In [1]:
import os
import findspark

# ── set hadoop home before starting spark ────────────────────────────
os.environ["HADOOP_HOME"]  = r"D:\hadoop"
os.environ["PATH"]         = r"D:\hadoop\bin;" + os.environ["PATH"]

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Airline_Ingestion") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.files.maxPartitionBytes", "128m") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("=" * 55)
print(f"✅ HADOOP_HOME         : {os.environ['HADOOP_HOME']}")
print(f"✅ Spark version       : {spark.version}")
print(f"✅ App name            : {spark.sparkContext.appName}")
print(f"✅ Driver memory       : 4 GB")
print(f"✅ Off heap memory     : 2 GB")
print("=" * 55)
print("🚀 Spark session ready!")

✅ HADOOP_HOME         : D:\hadoop
✅ Spark version       : 3.5.1
✅ App name            : Airline_Ingestion
✅ Driver memory       : 4 GB
✅ Off heap memory     : 2 GB
🚀 Spark session ready!


In [2]:
import os
from pyspark.sql.types import *

RAW_DIR       = r"D:\airline_pipeline\data\raw"
PROCESSED_DIR = r"D:\airline_pipeline\data\processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── get all csv file paths explicitly ────────────────────────────────
csv_files = [
    os.path.join(RAW_DIR, f) 
    for f in os.listdir(RAW_DIR) 
    if f.endswith(".csv")
]
print(f"📂 Found {len(csv_files)} CSV files")

# ── define schema manually ────────────────────────────────────────────
schema = StructType([
    StructField("Year",                            IntegerType(), True),
    StructField("Quarter",                         IntegerType(), True),
    StructField("Month",                           IntegerType(), True),
    StructField("DayofMonth",                      IntegerType(), True),
    StructField("DayOfWeek",                       IntegerType(), True),
    StructField("FlightDate",                      StringType(),  True),
    StructField("Reporting_Airline",               StringType(),  True),
    StructField("Flight_Number_Reporting_Airline", IntegerType(), True),
    StructField("Origin",                          StringType(),  True),
    StructField("OriginCityName",                  StringType(),  True),
    StructField("OriginState",                     StringType(),  True),
    StructField("Dest",                            StringType(),  True),
    StructField("DestCityName",                    StringType(),  True),
    StructField("DestState",                       StringType(),  True),
    StructField("CRSDepTime",                      IntegerType(), True),
    StructField("DepTime",                         DoubleType(),  True),
    StructField("DepDelay",                        DoubleType(),  True),
    StructField("DepDelayMinutes",                 DoubleType(),  True),
    StructField("CRSArrTime",                      IntegerType(), True),
    StructField("ArrTime",                         DoubleType(),  True),
    StructField("ArrDelay",                        DoubleType(),  True),
    StructField("ArrDelayMinutes",                 DoubleType(),  True),
    StructField("Cancelled",                       DoubleType(),  True),
    StructField("CancellationCode",                StringType(),  True),
    StructField("Diverted",                        DoubleType(),  True),
    StructField("Distance",                        DoubleType(),  True),
    StructField("CarrierDelay",                    DoubleType(),  True),
    StructField("WeatherDelay",                    DoubleType(),  True),
    StructField("NASDelay",                        DoubleType(),  True),
    StructField("SecurityDelay",                   DoubleType(),  True),
    StructField("LateAircraftDelay",               DoubleType(),  True),
])

# ── load using explicit file list instead of wildcard ─────────────────
print("📂 Loading CSV files into Spark...")

df_raw = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(csv_files)

print("=" * 55)
print(f"✅ Files loaded        : {len(csv_files)}")
print(f"✅ Total columns       : {len(df_raw.columns)}")
print(f"✅ Data size           : 11.4 GB")
print("=" * 55)
print("🚀 Data loaded! Ready for next step!")

📂 Found 47 CSV files
📂 Loading CSV files into Spark...
✅ Files loaded        : 47
✅ Total columns       : 31
✅ Data size           : 11.4 GB
🚀 Data loaded! Ready for next step!


In [4]:
print("📋 SCHEMA — Column names and data types:")
print("=" * 55)
df_raw.printSchema()

📋 SCHEMA — Column names and data types:
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: double (nullable = true)
 |-- Cancelled

In [5]:
from pyspark.sql import functions as F

# ── key columns we care about ─────────────────────────────────────────
key_cols = [
    "Year", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate", "Reporting_Airline",
    "Origin", "Dest",
    "DepDelay", "ArrDelay",
    "Cancelled", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "Distance"
]

# ── only check columns that exist ────────────────────────────────────
existing_cols = [c for c in key_cols if c in df_raw.columns]

print("🔍 NULL CHECK — Key columns:")
print("=" * 55)
print(f"{'Column':<30} {'Null %':>8}")
print("-" * 55)

null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in existing_cols]
null_counts = df_raw.select(null_exprs).collect()[0]

for col in existing_cols:
    count = null_counts[col]
    flag  = " ⚠️" if count and count > 1000000 else ""
    print(f"  {col:<28} {str(count):>10}{flag}")

print("=" * 55)
print("✅ Null check complete!")

🔍 NULL CHECK — Key columns:
Column                           Null %
-------------------------------------------------------
  Year                                  0
  Month                                 0
  DayofMonth                            0
  DayOfWeek                             0
  FlightDate                            0
  Reporting_Airline                     0
  Origin                                0
  Dest                                  0
  DepDelay                       27075400 ⚠️
  ArrDelay                              0
  Cancelled                             0
  Diverted                       27075400 ⚠️
  CarrierDelay                          0
  WeatherDelay                   27075400 ⚠️
  NASDelay                              0
  SecurityDelay                         0
  LateAircraftDelay                443976
  Distance                       27075400 ⚠️
✅ Null check complete!


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

print("🧹 Cleaning data...")

df_clean = df_raw

# ── 1. drop rows where core fields are null ───────────────────────────
df_clean = df_clean.dropna(subset=[
    "Year", "Month", "Origin", "Dest", "Reporting_Airline"
])

# ── 2. fill delay nulls with 0 ────────────────────────────────────────
df_clean = df_clean.fillna({
    "CarrierDelay"     : 0.0,
    "WeatherDelay"     : 0.0,
    "NASDelay"         : 0.0,
    "SecurityDelay"    : 0.0,
    "LateAircraftDelay": 0.0,
    "DepDelay"         : 0.0,
    "ArrDelay"         : 0.0,
    "DepDelayMinutes"  : 0.0,
    "ArrDelayMinutes"  : 0.0,
    "Distance"         : 0.0,
    "Diverted"         : 0.0,
})

# ── 3. fix data types ─────────────────────────────────────────────────
df_clean = df_clean \
    .withColumn("Year",      F.col("Year").cast(IntegerType())) \
    .withColumn("Month",     F.col("Month").cast(IntegerType())) \
    .withColumn("Distance",  F.col("Distance").cast(DoubleType())) \
    .withColumn("Cancelled", F.col("Cancelled").cast(IntegerType())) \
    .withColumn("DepDelay",  F.col("DepDelay").cast(DoubleType())) \
    .withColumn("ArrDelay",  F.col("ArrDelay").cast(DoubleType()))

# ── 4. remove bad rows ────────────────────────────────────────────────
df_clean = df_clean.filter(F.col("Year").between(2022, 2025))
df_clean = df_clean.filter(F.col("Distance") > 0)

# ── 5. add useful derived columns ────────────────────────────────────
df_clean = df_clean \
    .withColumn("IsDelayed",   (F.col("ArrDelay") > 15).cast(IntegerType())) \
    .withColumn("IsCancelled", (F.col("Cancelled") == 1).cast(IntegerType())) \
    .withColumn("TotalDelay",  
        F.col("CarrierDelay") + 
        F.col("WeatherDelay") + 
        F.col("NASDelay") + 
        F.col("SecurityDelay") + 
        F.col("LateAircraftDelay")
    )

print("=" * 55)
print(f"✅ Nulls filled         : all delay columns → 0")
print(f"✅ Bad rows removed     : distance = 0")
print(f"✅ Years kept           : 2022 → 2025")
print(f"✅ New columns added    : IsDelayed, IsCancelled, TotalDelay")
print("=" * 55)
print("🧹 Data cleaning complete!")

🧹 Cleaning data...
✅ Nulls filled         : all delay columns → 0
✅ Bad rows removed     : distance = 0
✅ Years kept           : 2022 → 2025
✅ New columns added    : IsDelayed, IsCancelled, TotalDelay
🧹 Data cleaning complete!


In [8]:
import time
import os

PARQUET_PATH = PROCESSED_DIR + r"\flights_clean.parquet"

print("💾 Saving cleaned data as Parquet...")
print("   partitioned by Year and Month")
print("   please wait — this takes 5-10 minutes\n")

start_time = time.time()

df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .parquet(PARQUET_PATH)

end_time = time.time()
duration = round((end_time - start_time) / 60, 2)

print("=" * 55)
print(f"✅ Parquet saved to:")
print(f"   {PARQUET_PATH}")
print(f"⏱️  Time taken : {duration} minutes")
print("=" * 55)

# ── verify by checking folder size ───────────────────────────────────
print("\n🔍 Verifying saved files...")

parquet_files = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for file in files:
        if file.endswith(".parquet"):
            parquet_files.append(os.path.join(root, file))

total_size = sum(os.path.getsize(f) for f in parquet_files) / (1024**3)

print(f"✅ Parquet files found  : {len(parquet_files)}")
print(f"✅ Total size on disk   : {total_size:.2f} GB")
print(f"✅ Partitioned by       : Year + Month")
print("=" * 55)
print("🎉 Ingestion complete! Ready for Step 6!")

💾 Saving cleaned data as Parquet...
   partitioned by Year and Month
   please wait — this takes 5-10 minutes

✅ Parquet saved to:
   D:\airline_pipeline\data\processed\flights_clean.parquet
⏱️  Time taken : 6.84 minutes

🔍 Verifying saved files...
✅ Parquet files found  : 0
✅ Total size on disk   : 0.00 GB
✅ Partitioned by       : Year + Month
🎉 Ingestion complete! Ready for Step 6!


In [9]:
import os

PROCESSED_DIR = r"D:\airline_pipeline\data\processed"

print("🔍 Checking processed folder contents...")
print("=" * 55)

for root, dirs, files in os.walk(PROCESSED_DIR):
    level = root.replace(PROCESSED_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for file in files[:3]:  # show first 3 files only
        size_mb = os.path.getsize(os.path.join(root, file)) / (1024*1024)
        print(f"{indent}  📄 {file} ({size_mb:.1f} MB)")
    if len(files) > 3:
        print(f"{indent}  ... and {len(files)-3} more files")

print("=" * 55)

🔍 Checking processed folder contents...
📁 processed/
  📁 flights_clean.parquet/
    📄 ._SUCCESS.crc (0.0 MB)
    📄 _SUCCESS (0.0 MB)


In [10]:
import os

PARQUET_PATH = r"D:\airline_pipeline\data\processed\flights_clean.parquet"

print("🔍 Deep scan of parquet folder...")
print("=" * 55)

all_files = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb   = os.path.getsize(full_path) / (1024*1024)
        all_files.append((full_path, size_mb))

parquet_files = [(p, s) for p, s in all_files if p.endswith(".parquet")]
total_size    = sum(s for _, s in parquet_files)

print(f"✅ Total files found    : {len(all_files)}")
print(f"✅ Parquet files        : {len(parquet_files)}")
print(f"✅ Total size           : {total_size:.2f} MB")

print(f"\n📁 First 5 parquet files:")
for path, size in parquet_files[:5]:
    short = path.replace(PARQUET_PATH, "")
    print(f"   {short}  ({size:.1f} MB)")

print("=" * 55)

🔍 Deep scan of parquet folder...
✅ Total files found    : 2
✅ Parquet files        : 0
✅ Total size           : 0.00 MB

📁 First 5 parquet files:


In [11]:
import time
import os

PARQUET_PATH = r"D:\airline_pipeline\data\processed\flights_clean.parquet"

# ── check df_clean still has data ────────────────────────────────────
print(f"✅ df_clean columns     : {len(df_clean.columns)}")
print(f"✅ Sample row check...")
df_clean.show(2)

✅ df_clean columns     : 34
✅ Sample row check...
+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+----------+-------+--------+---------------+---------+----------------+--------+--------+------------+------------+--------+-------------+-----------------+---------+-----------+----------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Flight_Number_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName|DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|Cancelled|CancellationCode|Diverted|Distance|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|IsDelayed|IsCancelled|TotalDelay|
+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+--------------+-----------+----+------------+

In [1]:
import os
import time
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── set hadoop ────────────────────────────────────────────────────────
os.environ["HADOOP_HOME"] = r"D:\hadoop"
os.environ["PATH"]        = r"D:\hadoop\bin;" + os.environ["PATH"]

findspark.init()

# ── start spark ───────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("Airline_Ingestion") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.files.maxPartitionBytes", "128m") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark started!")

# ── paths ─────────────────────────────────────────────────────────────
RAW_DIR       = r"D:\airline_pipeline\data\raw"
PROCESSED_DIR = r"D:\airline_pipeline\data\processed"
PARQUET_PATH  = PROCESSED_DIR + r"\flights_clean.parquet"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── schema ────────────────────────────────────────────────────────────
schema = StructType([
    StructField("Year",                            IntegerType(), True),
    StructField("Quarter",                         IntegerType(), True),
    StructField("Month",                           IntegerType(), True),
    StructField("DayofMonth",                      IntegerType(), True),
    StructField("DayOfWeek",                       IntegerType(), True),
    StructField("FlightDate",                      StringType(),  True),
    StructField("Reporting_Airline",               StringType(),  True),
    StructField("Flight_Number_Reporting_Airline", IntegerType(), True),
    StructField("Origin",                          StringType(),  True),
    StructField("OriginCityName",                  StringType(),  True),
    StructField("OriginState",                     StringType(),  True),
    StructField("Dest",                            StringType(),  True),
    StructField("DestCityName",                    StringType(),  True),
    StructField("DestState",                       StringType(),  True),
    StructField("CRSDepTime",                      IntegerType(), True),
    StructField("DepTime",                         DoubleType(),  True),
    StructField("DepDelay",                        DoubleType(),  True),
    StructField("DepDelayMinutes",                 DoubleType(),  True),
    StructField("CRSArrTime",                      IntegerType(), True),
    StructField("ArrTime",                         DoubleType(),  True),
    StructField("ArrDelay",                        DoubleType(),  True),
    StructField("ArrDelayMinutes",                 DoubleType(),  True),
    StructField("Cancelled",                       DoubleType(),  True),
    StructField("CancellationCode",                StringType(),  True),
    StructField("Diverted",                        DoubleType(),  True),
    StructField("Distance",                        DoubleType(),  True),
    StructField("CarrierDelay",                    DoubleType(),  True),
    StructField("WeatherDelay",                    DoubleType(),  True),
    StructField("NASDelay",                        DoubleType(),  True),
    StructField("SecurityDelay",                   DoubleType(),  True),
    StructField("LateAircraftDelay",               DoubleType(),  True),
])

# ── load CSVs ─────────────────────────────────────────────────────────
csv_files = [
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith(".csv")
]
print(f"✅ Found {len(csv_files)} CSV files")

df_raw = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(csv_files)

print(f"✅ Loaded {len(df_raw.columns)} columns")

# ── clean data ────────────────────────────────────────────────────────
df_clean = df_raw \
    .dropna(subset=["Year", "Month", "Origin", "Dest", "Reporting_Airline"]) \
    .fillna({
        "CarrierDelay": 0.0, "WeatherDelay": 0.0,
        "NASDelay": 0.0, "SecurityDelay": 0.0,
        "LateAircraftDelay": 0.0, "DepDelay": 0.0,
        "ArrDelay": 0.0, "DepDelayMinutes": 0.0,
        "ArrDelayMinutes": 0.0, "Distance": 0.0,
        "Diverted": 0.0,
    }) \
    .withColumn("Year",      F.col("Year").cast(IntegerType())) \
    .withColumn("Month",     F.col("Month").cast(IntegerType())) \
    .withColumn("Distance",  F.col("Distance").cast(DoubleType())) \
    .withColumn("Cancelled", F.col("Cancelled").cast(IntegerType())) \
    .withColumn("DepDelay",  F.col("DepDelay").cast(DoubleType())) \
    .withColumn("ArrDelay",  F.col("ArrDelay").cast(DoubleType())) \
    .filter(F.col("Year").between(2022, 2025)) \
    .filter(F.col("Distance") > 0) \
    .withColumn("IsDelayed",   (F.col("ArrDelay") > 15).cast(IntegerType())) \
    .withColumn("IsCancelled", (F.col("Cancelled") == 1).cast(IntegerType())) \
    .withColumn("TotalDelay",
        F.col("CarrierDelay") + F.col("WeatherDelay") +
        F.col("NASDelay") + F.col("SecurityDelay") +
        F.col("LateAircraftDelay")
    )

print("✅ Data cleaned!")

# ── quick row check before saving ────────────────────────────────────
sample = df_clean.limit(5).collect()
print(f"✅ Sample rows check   : {len(sample)} rows visible")
print(f"✅ Total columns       : {len(df_clean.columns)}")

# ── save as parquet ───────────────────────────────────────────────────
print("\n💾 Saving as Parquet — please wait 5-10 minutes...")
start = time.time()

df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .parquet(PARQUET_PATH)

duration = round((time.time() - start) / 60, 2)
print(f"✅ Saved in {duration} minutes!")

# ── verify ────────────────────────────────────────────────────────────
all_parquet = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for f in files:
        if f.endswith(".parquet"):
            all_parquet.append(os.path.join(root, f))

total_gb = sum(os.path.getsize(f) for f in all_parquet) / (1024**3)

print("=" * 55)
print(f"✅ Parquet files saved  : {len(all_parquet)}")
print(f"✅ Total size on disk   : {total_gb:.2f} GB")
print("=" * 55)
print("🎉 Ingestion complete! Ready for Step 6!")

✅ Spark started!
✅ Found 47 CSV files
✅ Loaded 31 columns
✅ Data cleaned!
✅ Sample rows check   : 0 rows visible
✅ Total columns       : 34

💾 Saving as Parquet — please wait 5-10 minutes...
✅ Saved in 4.96 minutes!
✅ Parquet files saved  : 0
✅ Total size on disk   : 0.00 GB
🎉 Ingestion complete! Ready for Step 6!


In [2]:
import os
from pyspark.sql.types import *

os.environ["HADOOP_HOME"] = r"D:\hadoop"
os.environ["PATH"]        = r"D:\hadoop\bin;" + os.environ["PATH"]

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("debug") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

RAW_DIR = r"D:\airline_pipeline\data\raw"

# ── load just ONE file to inspect ────────────────────────────────────
csv_files = [
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith(".csv")
]

print(f"✅ Loading just 1 file to debug...")
print(f"   File: {csv_files[0]}\n")

# load without schema to see raw values
df_test = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_files[0])

print(f"✅ Columns found: {df_test.columns}\n")
print(f"✅ First 5 rows:")
df_test.select("Year", "Month", "Origin", "Dest", "ArrDelay").show(5)

print(f"\n✅ Distinct Year values:")
df_test.select("Year").distinct().show()

✅ Loading just 1 file to debug...
   File: D:\airline_pipeline\data\raw\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv

✅ Columns found: ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'A

In [1]:
import os
import time
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# ── hadoop setup ──────────────────────────────────────────────────────
os.environ["HADOOP_HOME"] = r"D:\hadoop"
os.environ["PATH"]        = r"D:\hadoop\bin;" + os.environ["PATH"]
findspark.init()

# ── start spark ───────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("Airline_Ingestion") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.files.maxPartitionBytes", "128m") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark started!")

# ── paths ─────────────────────────────────────────────────────────────
RAW_DIR       = r"D:\airline_pipeline\data\raw"
PROCESSED_DIR = r"D:\airline_pipeline\data\processed"
PARQUET_PATH  = PROCESSED_DIR + r"\flights_clean.parquet"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── get file list ─────────────────────────────────────────────────────
csv_files = [
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith(".csv")
]
print(f"✅ Found {len(csv_files)} CSV files")

# ── load with inferSchema on ONE file first to get schema ─────────────
print("⏳ Reading schema from first file...")
df_schema = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_files[0])

print(f"✅ Total columns in CSV : {len(df_schema.columns)}")

# ── now load ALL files using that schema ──────────────────────────────
print("⏳ Loading all 47 CSV files...")
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_files)

# ── select only the columns we need ──────────────────────────────────
cols_we_need = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate", "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes",
    "Cancelled", "CancellationCode", "Diverted", "Distance",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

# keep only columns that exist
existing = [c for c in cols_we_need if c in df_raw.columns]
df_raw   = df_raw.select(existing)
print(f"✅ Selected {len(existing)} columns")

# ── clean ─────────────────────────────────────────────────────────────
df_clean = df_raw \
    .dropna(subset=["Year", "Month", "Origin", "Dest", "Reporting_Airline"]) \
    .fillna({
        "CarrierDelay": 0.0, "WeatherDelay": 0.0,
        "NASDelay": 0.0, "SecurityDelay": 0.0,
        "LateAircraftDelay": 0.0, "DepDelay": 0.0,
        "ArrDelay": 0.0, "DepDelayMinutes": 0.0,
        "ArrDelayMinutes": 0.0, "Distance": 0.0,
        "Diverted": 0.0,
    }) \
    .withColumn("Year",      F.col("Year").cast(IntegerType())) \
    .withColumn("Month",     F.col("Month").cast(IntegerType())) \
    .withColumn("Distance",  F.col("Distance").cast(DoubleType())) \
    .withColumn("Cancelled", F.col("Cancelled").cast(IntegerType())) \
    .withColumn("DepDelay",  F.col("DepDelay").cast(DoubleType())) \
    .withColumn("ArrDelay",  F.col("ArrDelay").cast(DoubleType())) \
    .filter(F.col("Year").between(2022, 2025)) \
    .filter(F.col("Distance") > 0) \
    .withColumn("IsDelayed",   (F.col("ArrDelay") > 15).cast(IntegerType())) \
    .withColumn("IsCancelled", (F.col("Cancelled") == 1).cast(IntegerType())) \
    .withColumn("TotalDelay",
        F.col("CarrierDelay") + F.col("WeatherDelay") +
        F.col("NASDelay") + F.col("SecurityDelay") +
        F.col("LateAircraftDelay")
    )

print("✅ Data cleaned!")

# ── quick check ───────────────────────────────────────────────────────
print("\n🔍 Quick sample check:")
df_clean.select("Year", "Month", "Origin", "Dest", "ArrDelay").show(5)

# ── save ──────────────────────────────────────────────────────────────
print("💾 Saving as Parquet — please wait 5-10 minutes...")
start = time.time()

df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .parquet(PARQUET_PATH)

duration = round((time.time() - start) / 60, 2)

# ── verify ────────────────────────────────────────────────────────────
all_parquet = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for f in files:
        if f.endswith(".parquet"):
            all_parquet.append(os.path.join(root, f))

total_gb = sum(os.path.getsize(f) for f in all_parquet) / (1024**3)

print("=" * 55)
print(f"✅ Saved in            : {duration} minutes")
print(f"✅ Parquet files       : {len(all_parquet)}")
print(f"✅ Total size on disk  : {total_gb:.2f} GB")
print("=" * 55)
print("🎉 Ingestion complete! Ready for Step 6!")

✅ Spark started!
✅ Found 47 CSV files
⏳ Reading schema from first file...
✅ Total columns in CSV : 110
⏳ Loading all 47 CSV files...
✅ Selected 31 columns
✅ Data cleaned!

🔍 Quick sample check:
+----+-----+------+----+--------+
|Year|Month|Origin|Dest|ArrDelay|
+----+-----+------+----+--------+
|2022|    1|   CMH| DCA|     4.0|
|2022|    1|   CMH| DCA|   -24.0|
|2022|    1|   CMH| DCA|   -13.0|
|2022|    1|   CMH| DCA|     9.0|
|2022|    1|   CMH| DCA|   -29.0|
+----+-----+------+----+--------+
only showing top 5 rows

💾 Saving as Parquet — please wait 5-10 minutes...
✅ Saved in            : 1.23 minutes
✅ Parquet files       : 109
✅ Total size on disk  : 0.39 GB
🎉 Ingestion complete! Ready for Step 6!


In [2]:
spark.stop()
print("✅ Spark stopped cleanly!")
print("\n📊 Step 5 Summary:")
print("=" * 55)
print("✅ Source       : 47 CSV files (11.4 GB)")
print("✅ Output       : Parquet format (0.39 GB)")
print("✅ Files saved  : 109 parquet files")
print("✅ Partitioned  : by Year and Month")
print("✅ Columns      : 31 + 3 derived columns")
print("✅ Years        : 2022, 2023, 2024, 2025")
print("=" * 55)
print("🚀 Ready for Step 6 — Spark Processing!")

✅ Spark stopped cleanly!

📊 Step 5 Summary:
✅ Source       : 47 CSV files (11.4 GB)
✅ Output       : Parquet format (0.39 GB)
✅ Files saved  : 109 parquet files
✅ Partitioned  : by Year and Month
✅ Columns      : 31 + 3 derived columns
✅ Years        : 2022, 2023, 2024, 2025
🚀 Ready for Step 6 — Spark Processing!


In [1]:
import os
import sys

# ── windows configuration (matching your lab setup) ───────────────────
os.environ["JAVA_HOME"]            = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.10-hotspot"
os.environ["HADOOP_HOME"]          = r"D:\hadoop"
os.environ["PATH"]                 += os.pathsep + r"D:\hadoop\bin"
os.environ["PYSPARK_PYTHON"]       = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"]= sys.executable
os.environ["SPARK_LOCAL_IP"]       = "127.0.0.1"

print("✅ Windows Environment Configured!")
print(f"   JAVA_HOME    : {os.environ['JAVA_HOME']}")
print(f"   HADOOP_HOME  : {os.environ['HADOOP_HOME']}")
print(f"   Python       : {sys.executable}")

✅ Windows Environment Configured!
   JAVA_HOME    : C:\Program Files\Eclipse Adoptium\jdk-17.0.18.10-hotspot
   HADOOP_HOME  : D:\hadoop
   Python       : C:\Users\easho\anaconda3\envs\pyspark_env\python.exe


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Airline_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark version : {spark.version}")
print(f"✅ Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"✅ Cores         : {spark.sparkContext.defaultParallelism}")
print("🚀 Spark is running!")

In [1]:
import os
import sys

# ── check java ────────────────────────────────────────────────────────
java_path = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.10-hotspot"
java_exe  = os.path.join(java_path, "bin", "java.exe")

print("=" * 55)
print("SYSTEM DIAGNOSTIC")
print("=" * 55)

# check java
if os.path.exists(java_exe):
    print(f"✅ Java found      : {java_exe}")
else:
    print(f"❌ Java NOT found  : {java_exe}")
    print("   Looking for Java in other locations...")
    for drive in ["C:", "D:"]:
        for root, dirs, files in os.walk(drive + "\\Program Files"):
            for f in files:
                if f == "java.exe":
                    print(f"   Found at: {os.path.join(root, f)}")
            # don't go too deep
            if root.count(os.sep) > 6:
                break

# check hadoop
hadoop_path = r"D:\hadoop\bin\winutils.exe"
if os.path.exists(hadoop_path):
    print(f"✅ Winutils found  : {hadoop_path}")
else:
    print(f"❌ Winutils NOT found : {hadoop_path}")

# check python
print(f"✅ Python          : {sys.executable}")
print(f"✅ Python version  : {sys.version}")

# check pyspark
try:
    import pyspark
    print(f"✅ PySpark         : {pyspark.__version__}")
except:
    print(f"❌ PySpark NOT installed!")

print("=" * 55)

SYSTEM DIAGNOSTIC
❌ Java NOT found  : C:\Program Files\Eclipse Adoptium\jdk-17.0.18.10-hotspot\bin\java.exe
   Looking for Java in other locations...
✅ Winutils found  : D:\hadoop\bin\winutils.exe
✅ Python          : C:\Users\easho\anaconda3\envs\pyspark_env\python.exe
✅ Python version  : 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
✅ PySpark         : 4.1.1


In [2]:
import os
import sys

os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"]           = r"D:\hadoop"
os.environ["PATH"]                  += os.pathsep + r"D:\hadoop\bin"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"

print("✅ Environment configured!")
print(f"   JAVA_HOME   : {os.environ['JAVA_HOME']}")
print(f"   HADOOP_HOME : {os.environ['HADOOP_HOME']}")
print(f"   Python      : {sys.executable}")

✅ Environment configured!
   JAVA_HOME   : C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
   HADOOP_HOME : D:\hadoop
   Python      : C:\Users\easho\anaconda3\envs\pyspark_env\python.exe


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Airline_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark version : {spark.version}")
print(f"✅ Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"✅ Cores         : {spark.sparkContext.defaultParallelism}")
print("🚀 Spark is running!")

✅ Spark version : 4.1.1
✅ Spark UI      : http://view-localhost:4040
✅ Cores         : 12
🚀 Spark is running!


In [4]:
import os

PARQUET_PATH = r"D:\airline_pipeline\data\processed\flights_clean.parquet"

all_parquet = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for f in files:
        if f.endswith(".parquet"):
            all_parquet.append(os.path.join(root, f))

total_gb = sum(os.path.getsize(f) for f in all_parquet) / (1024**3)

print(f"✅ Parquet files found : {len(all_parquet)}")
print(f"✅ Total size          : {total_gb:.2f} GB")

if len(all_parquet) > 0:
    print("🎉 Data is ready! Skip to Step 6!")
else:
    print("❌ No parquet files — need to rerun ingestion")

✅ Parquet files found : 109
✅ Total size          : 0.39 GB
🎉 Data is ready! Skip to Step 6!
